In [6]:
from google import genai
from google.genai import types
import pandas as pd
import google.generativeai as genai
import json
import os.path

import gspread
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from datetime import datetime, timedelta

def main():
    df1 = extract_financials_with_gemini(r"C:\Users\mochamad.ilmawan\OneDrive - semenindonesia.com\Renstra\Dashboard KPI Financial Aspect\Dashboard KPI Renstra_send.xlsx")
    df2 = extract_financials_with_gemini2(r"C:\Users\mochamad.ilmawan\OneDrive - semenindonesia.com\Renstra\Dashboard KPI Financial Aspect\Dashboard KPI Renstra_send.xlsx")
    
    print("DF1 Preview:\n", df1.head() if df1 is not None else "None")
    print("DF2 Preview:\n", df2.head() if df2 is not None else "None")

    # Safely combine both dataframes
    if df1 is not None and df2 is not None:
        df = pd.concat([df1, df2], axis=0, ignore_index=True)
    elif df1 is not None:
        df = df1
    else:
        df = df2

    if df is not None and not df.empty:
        print("Data combined and extracted successfully.")
        creds = connect_to_sheet()
        update_sheet('1taze3KNq8g1U76oSjp1yN_jMu6lzCZJiin0BQX5LzTs', 'COGS Variance 2!A:Z', None, creds, df)
    else:
        print("Failed to extract data. Check Gemini response or JSON format.")

def connect_to_sheet():
    SCOPES = ['https://www.googleapis.com/auth/spreadsheets']
    creds = None
    if os.path.exists(r'C:\Users\mochamad.ilmawan\token.json'):
        creds = Credentials.from_authorized_user_file(r'C:\Users\mochamad.ilmawan\token.json', SCOPES)
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(
                r'D:\OneDrive\Documents\Direct Selling\MONITORING\Monitoring 2023\client_secret_738271294618-6g8e0hle4jpq8nau1d7b3q9jfsgp0ruk.apps.googleusercontent.com.json', SCOPES)
            creds = flow.run_local_server(port=0)
        with open(r'C:\Users\mochamad.ilmawan\token.json', 'w') as token:
            token.write(creds.to_json())
    return creds

def get_excel_data(address, sheet_name=0, usecols=None, skiprows=0, nrows=None, header=None):
    df = pd.read_excel(address, sheet_name=sheet_name, usecols=usecols, skiprows=skiprows, nrows=nrows, dtype=str, thousands='.', decimal=',')
    return df

def update_sheet(spreadsheetId, spread_range, scopes, creds, df):
    df_clean = df.fillna('')
    header = df_clean.columns.tolist()
    data = df_clean.values.tolist()
    values_with_header = [header] + data
    try:
        # FIXED: Changed service.sheets() to service.spreadsheets()
        service = build('sheets', 'v4', credentials=creds)
        service.spreadsheets().values().clear(spreadsheetId=spreadsheetId, range=spread_range, body={}).execute()
        
        body = {'values': values_with_header}
        result = service.spreadsheets().values().update(
            spreadsheetId=spreadsheetId, range=spread_range, valueInputOption="USER_ENTERED", body=body).execute()
        print(f"{result.get('updatedCells')} cells updated (sheet replaced).")
        return result
    except HttpError as error:
        print(f"An error occurred: {error}")
        return error

def extract_financials_with_gemini2(file_path):
    genai.configure(api_key="AQ.Ab8RN6K-C_Hpr3I3ra-94nf7F76ILVJFUXD4r61dpwelFN3BcA")
    generation_config = {
        "temperature": 0,
        "top_p": 0.95,
        "response_mime_type": "application/json",
    }
    model = genai.GenerativeModel(model_name="gemini-3.5-flash", generation_config=generation_config)
    
    df_rkap = get_excel_data(file_path, sheet_name='RKAP', usecols="C:N", skiprows=1, nrows=61, header=2)
    df_real = get_excel_data(file_path, sheet_name='REAL', usecols="C:N", skiprows=1, nrows=61, header=2)
    
    combined_raw_data = f"=== RAW DATA FROM RKAP SHEET ===\n{df_rkap.to_string()}\n\n=== RAW DATA FROM REAL SHEET ===\n{df_real.to_string()}"
    
    prompt = f"""
    Act as a precise financial data extraction engine. The provided data tables contain monthly columns spanning from January to December. Your task is to unpivot this data so that each metric has 12 separate entries (one for each month of the year 2026).

    ### EXTRACTION RULES:
    1. TIMELINE EXTRACTION: For every item in the METRIC LIST, extract its values for all 12 months (January, February, March, ..., December).
    2. MONTH FORMAT: Represent the 'Month' field as the last day of that specific month in 'DD/MM/YYYY' format for the year 2026 (e.g., Jan = 31/01/2026, Feb = 28/02/2026, Mar = 31/03/2026, etc.).
    3. VALUE MAPPING:
       - Match the correct month column from the RKAP sheet to populate 'Volume_RKAP' and 'Harga_RKAP'.
       - Match the correct month column from the REAL sheet to populate 'Volume_Real' and 'Harga_Real'.
       - If a value is missing, empty, or "-" for a specific month, use 0.0.
    4. GROUP ASSIGNMENT: Assign the correct category ('Bahan Baku', 'Bahan Bakar', 'Listrik', 'Kemasan', 'Perniagaan', or 'Other').
    5. TOTAL EXTRACTION: Find the row data for "PROD CEMENT" and "PROD CLINKER". Extract their values per month and assign them to fields 'Total_Volume_RKAP' and 'Total_Volume_Real' dynamically based on which production group the asset belongs to.
    6. OUTPUT: Return a flat JSON array of objects. Do not truncate the list.

    ### METRIC LIST:
    3. Batu kapur
    4. Tanah Liat
    5. Pasir Silika
    6. Pasir Besi & Copper Slag
    7. Trass Pozolan
    8. Gypsum
    9. Fly Ash
    10. CGA
    11. Bahan Bakar
    12. Batubara
    13. Solar
    14. Sekam Padi
    15. Upto Kiln
    16. FM
    17. Others
    18. Kemasan

    ### REQUIRED JSON STRUCTURE EXAMPLE:
    [
      {{
        "Account": "Batu kapur",
        "Group": "Bahan Baku",
        "Month": "31/01/2026",
        "Volume_RKAP": 120000.0,
        "Volume_Real": 115000.0,
        "Harga_RKAP": 28000.0,
        "Harga_Real": 26000.0,
        "Total_Volume_RKAP": 2461795,
        "Total_Volume_Real": 2214589
      }}
    ]

    Raw Input Text Data:
    {combined_raw_data}
    """

    try:
        response = model.generate_content(prompt, request_options={"timeout": 300.0})
        json_text = response.text.replace('```json', '').replace('```', '').strip()
        data = json.loads(json_text)
        return pd.DataFrame(data)
    except Exception as e:
        print(f"Error executing or parsing Gemini 2: {e}")
        return None

def extract_financials_with_gemini(file_path):
    genai.configure(api_key="AQ.Ab8RN6K-C_Hpr3I3ra-94nf7F76ILVJFUXD4r61dpwelFN3BcA")
    generation_config = {
        "temperature": 0,
        "top_p": 0.95,
        "response_mime_type": "application/json",
    }
    model = genai.GenerativeModel(model_name="gemini-3.5-flash", generation_config=generation_config)
    
    # Make sure usecols grabs the columns that contain the 12 months (e.g., C to AD or whichever columns hold the monthly trends)
    df_raw = get_excel_data(file_path, 'KPI Aspek Keuangan', "C:AD", 1, 99, 2)
    raw_data_string = df_raw.to_string()

    prompt = f"""
    Act as a precise financial data extraction engine. The provided data table contains monthly columns spanning from January to December. Your task is to unpivot this data so that each metric has 12 separate entries (one for each month of the year 2026).

    ### EXTRACTION RULES:
    1. TIMELINE EXTRACTION: For every item in the METRIC LIST, extract its values for all 12 months (January, February, March, ..., December).
    2. MONTH FORMAT: Represent the 'Month' field as the last day of that specific month in 'DD/MM/YYYY' format for the year 2026 (e.g., Jan = 31/01/2026, Feb = 28/02/2026, Mar = 31/03/2026, etc.).
    3. VALUE MAPPING:
       - Locate the monthly columns in the raw data to populate 'Harga_RKAP' and 'Harga_Real' for each specific month.
       - If a value is missing, empty, or "-" for a specific month, use 0.0.
    4. GROUP ASSIGNMENT: Assign the correct category ('Pemeliharaan', 'Perniagaan', or 'Other').
    5. DEFAULT VALUES: Set 'Volume_RKAP' and 'Volume_Real' to 1 for all rows.
    6. TOTAL EXTRACTION: Find the row data for "VOLUME PENJUALAN (TON) - semen & klinker only". Extract its values per month and assign them to fields 'Total_Volume_RKAP' and 'Total_Volume_Real' dynamically for each month.
    7. OUTPUT: Return a flat JSON array of objects. Do not truncate the list.

    ### METRIC LIST:
    1. Pemeliharaan
    2. Urusan Umum & Adm. Kantor
    3. Perniagaan
    4. Selisih Persediaan

    ### REQUIRED JSON STRUCTURE EXAMPLE:
    [
      {{
        "Account": "Pemeliharaan",
        "Group": "Other",
        "Month": "31/01/2026",
        "Volume_RKAP": 1,
        "Volume_Real": 1,
        "Harga_RKAP": 153594.89,
        "Harga_Real": 148747.78,
        "Total_Volume_RKAP": 3044916.0,
        "Total_Volume_Real": 3282482.4
      }}
    ]

    Raw Data:
    {raw_data_string}
    """

    try:
        response = model.generate_content(prompt, request_options={"timeout": 120.0})
        json_text = response.text.replace('```json', '').replace('```', '').strip()
        data = json.loads(json_text)
        return pd.DataFrame(data)
    except Exception as e:
        print(f"Error executing or parsing Gemini 1: {e}")
        return None

if __name__ == '__main__':
    main()

DF1 Preview:
         Account         Group       Month  Volume_RKAP  Volume_Real  \
0  Pemeliharaan  Pemeliharaan  31/01/2026            1            1   
1  Pemeliharaan  Pemeliharaan  28/02/2026            1            1   
2  Pemeliharaan  Pemeliharaan  31/03/2026            1            1   
3  Pemeliharaan  Pemeliharaan  30/04/2026            1            1   
4  Pemeliharaan  Pemeliharaan  31/05/2026            1            1   

   Harga_RKAP  Harga_Real  Total_Volume_RKAP  Total_Volume_Real  
0    153594.9   148747.78         3044916.31         3282482.41  
1    153594.9   142133.32         3225760.81         2964284.23  
2    153594.9   144484.06         2498010.62         2463530.37  
3    153594.9   145490.16         2981033.37         3192815.99  
4    153594.9   149090.14         3202622.94         3212870.88  
DF2 Preview:
       Account       Group       Month   Volume_RKAP   Volume_Real  \
0  Batu kapur  Bahan Baku  31/01/2026  3.999846e+06  3.661289e+06   
1  Batu kap